PHASE 7 — CROSS-MODEL REPLICATION

Model:
EleutherAI/pythia-160m

Question:
Does interaction-aware circuit selection outperform first-order
AIE selection at equal patch budget k=2 on a different Transformer?

Discovery:
32 new CounterFact facts.

Held-out validation:
128 additional subject-disjoint facts.

Primary endpoint:
delta = recovery_INTERACTION - recovery_AIE

Primary success:
bootstrap 95% CI lower bound > 0

Practical success:
mean delta >= 0.02

No changes to metric, corruption scheme, or success threshold
after held-out evaluation begins.

In [1]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from datasets import load_dataset

C:\Users\Mauro\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "EleutherAI/pythia-160m"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
).to(DEVICE).eval()

print(type(model))

N_LAYERS = len(
    model.gpt_neox.layers
)

EMBED_STD = (
    model.gpt_neox.embed_in
    .weight
    .detach()
    .std()
    .item()
)

print("layers:", N_LAYERS)
print("embedding std:", EMBED_STD)

device: cpu


C:\Users\Mauro\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


<class 'transformers.models.gpt_neox.modeling_gpt_neox.GPTNeoXForCausalLM'>
layers: 12
embedding std: 0.02854427695274353


In [3]:
counterfact = load_dataset(
    "azhx/counterfact",
    split="test"
)

print(counterfact)

Dataset({
    features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
    num_rows: 2191
})


In [4]:
phase5_seen = pd.read_csv(
    "phase5_seen_facts.csv"
)

phase6a_seen = pd.read_csv(
    "phase6_fresh_manifest.csv"
)

phase6b_seen = pd.read_csv(
    "phase6b_locked_manifest.csv"
)

def norm_subject(x):
    return str(x).strip().lower()

SUBJ5 = set(
    phase5_seen["subject"].map(norm_subject)
)

SUBJ6A = set(
    phase6a_seen["subject"].map(norm_subject)
)

SUBJ6B = set(
    phase6b_seen["subject"].map(norm_subject)
)

PREVIOUS_SUBJECTS = (
    SUBJ5
    | SUBJ6A
    | SUBJ6B
)

print("Phase 5:", len(SUBJ5))
print("Phase 6A:", len(SUBJ6A))
print("Phase 6B:", len(SUBJ6B))

print(
    "Total previous unique subjects:",
    len(PREVIOUS_SUBJECTS)
)

Phase 5: 64
Phase 6A: 128
Phase 6B: 128
Total previous unique subjects: 314


In [5]:
N_DISCOVERY = 32
N_HOLDOUT = 128

DISCOVERY_SEED = 20260911
HOLDOUT_SEED = 20260912

REFERENCE_SEED = 424242
REFERENCE_SCALE = 3.0

PRIMARY_SCALE = 3.0

NOISE_SCALES = [
    2.5,
    3.0,
    3.5
]

NOISE_SEEDS = [
    101,
    202,
    303
]

MIN_CLEAN_MARGIN = 1.0
MIN_REFERENCE_TE = 1.0
MIN_EVAL_TE = 0.5

In [6]:
def one_token_id(text):

    ids = tokenizer.encode(
        " " + text.strip(),
        add_special_tokens=False
    )

    if len(ids) != 1:
        return None

    return ids[0]


def subject_token_positions(
    prompt,
    subject
):

    enc = tokenizer(
        prompt,
        return_offsets_mapping=True,
        add_special_tokens=False
    )

    start = prompt.find(subject)

    if start < 0:
        return []

    end = start + len(subject)

    positions = []

    for i, (a, b) in enumerate(
        enc["offset_mapping"]
    ):

        if b > start and a < end:
            positions.append(i)

    return positions


def logit_margin(
    logits,
    true_id,
    alt_id
):

    return (
        logits[0, -1, true_id]
        -
        logits[0, -1, alt_id]
    ).item()

In [7]:
def corrupted_embeddings(
    input_ids,
    subj_positions,
    seed,
    noise_multiplier
):

    embeds = (
        model.gpt_neox.embed_in(
            input_ids
        )
        .detach()
        .clone()
    )

    g = torch.Generator(
        device=embeds.device
    )

    g.manual_seed(
        int(seed)
    )

    noise = torch.randn(
        embeds.shape,
        generator=g,
        device=embeds.device,
        dtype=embeds.dtype
    )

    noise *= (
        noise_multiplier
        * EMBED_STD
    )

    for pos in subj_positions:

        embeds[:, pos, :] += (
            noise[:, pos, :]
        )

    return embeds

In [8]:
def prepare_case(
    record,
    dataset_index
):

    rr = record[
        "requested_rewrite"
    ]

    subject = rr["subject"]
    prompt = rr["prompt"].format(subject)

    true_text = rr["target_true"]["str"]
    new_text = rr["target_new"]["str"]

    true_id = one_token_id(true_text)
    new_id = one_token_id(new_text)

    if (
        true_id is None
        or new_id is None
        or true_id == new_id
    ):
        return None

    subj_positions = (
        subject_token_positions(
            prompt,
            subject
        )
    )

    if not subj_positions:
        return None

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.inference_mode():

        clean_out = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    D_clean = logit_margin(
        clean_out.logits,
        true_id,
        new_id
    )

    if D_clean < MIN_CLEAN_MARGIN:
        return None

    ref_emb = corrupted_embeddings(
        input_ids,
        subj_positions,
        REFERENCE_SEED + int(dataset_index),
        REFERENCE_SCALE
    )

    with torch.inference_mode():

        ref_out = model(
            inputs_embeds=ref_emb,
            attention_mask=attention_mask
        )

    D_ref = logit_margin(
        ref_out.logits,
        true_id,
        new_id
    )

    TE_ref = D_clean - D_ref

    if TE_ref < MIN_REFERENCE_TE:
        return None

    return {
        "case_id": int(record["case_id"]),
        "dataset_index": int(dataset_index),
        "subject": subject,
        "prompt": prompt,
        "true_text": true_text,
        "new_text": new_text,
        "true_id": int(true_id),
        "new_id": int(new_id),
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "subject_positions": subj_positions,
        "subject_last": subj_positions[-1],
        "query_pos": input_ids.shape[1] - 1,
        "D_clean": float(D_clean),
        "TE_reference": float(TE_ref)
    }

In [9]:
rng = np.random.default_rng(
    DISCOVERY_SEED
)

indices = rng.permutation(
    len(counterfact)
)

discovery_cases = []
selected_subjects = set()

for idx in indices:

    record = counterfact[int(idx)]

    subject_key = norm_subject(
        record[
            "requested_rewrite"
        ]["subject"]
    )

    if subject_key in PREVIOUS_SUBJECTS:
        continue

    if subject_key in selected_subjects:
        continue

    case = prepare_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    discovery_cases.append(case)
    selected_subjects.add(subject_key)

    print(
        len(discovery_cases),
        case["subject"],
        "| TE =",
        round(
            case["TE_reference"],
            3
        )
    )

    if len(discovery_cases) >= N_DISCOVERY:
        break

1 Julien Courbet | TE = 2.538
2 Windows Mobile 6.5 | TE = 1.111
3 Brisbane Airport | TE = 4.155
4 Francis X. DiLorenzo | TE = 2.319
5 Nazi Party | TE = 1.217
6 Galina Starovoytova | TE = 2.588
7 Charles-Joseph Panckoucke | TE = 1.229
8 Saint-Jacques-de-la-Lande | TE = 3.25
9 The Kathmandu Post | TE = 1.704
10 Economic Community of West African States | TE = 1.077
11 Virginia | TE = 2.054
12 Broadway Open House | TE = 2.226
13 Microsoft Project | TE = 1.67
14 R. H. Bruce Lockhart | TE = 1.567
15 Honolulu International Airport | TE = 2.091
16 Hurrem sultan | TE = 1.303
17 New Zealand Labour Party | TE = 3.65
18 1977 Moscow bombings | TE = 3.025
19 Sommariva del Bosco | TE = 2.069
20 Google Person Finder | TE = 5.288
21 New Club, Edinburgh | TE = 4.059
22 Boeing Defense, Space & Security | TE = 2.642
23 Thomas Beaufort, Duke of Exeter | TE = 1.792
24 Michelangelo | TE = 1.796
25 Jerusalem syndrome | TE = 3.143
26 Montserrat Lombard | TE = 1.206
27 Bobby Taylor & the Vancouvers | TE = 1.17

In [10]:
assert len(discovery_cases) == N_DISCOVERY
assert len(selected_subjects) == N_DISCOVERY

assert (
    selected_subjects
    & PREVIOUS_SUBJECTS
) == set()

print(
    "Discovery cases:",
    len(discovery_cases)
)

print(
    "Overlap:",
    len(
        selected_subjects
        & PREVIOUS_SUBJECTS
    )
)

Discovery cases: 32
Overlap: 0


In [11]:
discovery_manifest = pd.DataFrame([
    {
        "case_id": c["case_id"],
        "dataset_index": c["dataset_index"],
        "subject": c["subject"],
        "prompt": c["prompt"],
        "true_text": c["true_text"],
        "new_text": c["new_text"],
        "D_clean": c["D_clean"],
        "TE_reference": c["TE_reference"]
    }
    for c in discovery_cases
])

discovery_manifest.to_csv(
    "phase7_discovery_manifest.csv",
    index=False
)

In [12]:
SITES = [
    (layer, role)
    for layer in range(N_LAYERS)
    for role in ["S", "Q"]
]

print("sites:", len(SITES))

sites: 24


In [13]:
def patched_margin(
    case,
    clean_states,
    corrupt_emb,
    sites
):

    patches_by_layer = {}

    for layer, role in sites:
        patches_by_layer.setdefault(
            layer, []
        ).append(role)

    handles = []

    for layer, roles in patches_by_layer.items():

        def make_hook(
            layer_idx,
            roles_here
        ):

            def hook(
                module,
                inputs,
                output
            ):

                if isinstance(output, tuple):
                    hidden = output[0].clone()
                else:
                    hidden = output.clone()

                for role in roles_here:

                    if role == "S":
                        pos = case["subject_last"]

                    elif role == "Q":
                        pos = case["query_pos"]

                    else:
                        raise ValueError(role)

                    clean_value = (
                        clean_states[
                            layer_idx + 1
                        ][:, pos, :]
                    )

                    hidden[:, pos, :] = clean_value

                if isinstance(output, tuple):
                    return (
                        hidden,
                        *output[1:]
                    )

                return hidden

            return hook

        h = (
            model.gpt_neox.layers[layer]
            .register_forward_hook(
                make_hook(
                    layer,
                    roles
                )
            )
        )

        handles.append(h)

    try:

        with torch.inference_mode():

            out = model(
                inputs_embeds=corrupt_emb,
                attention_mask=
                    case["attention_mask"]
            )

        return logit_margin(
            out.logits,
            case["true_id"],
            case["new_id"]
        )

    finally:

        for h in handles:
            h.remove()

In [14]:
# Sanity check
case = discovery_cases[0]

with torch.inference_mode():

    clean_out = model(
        input_ids=case["input_ids"],
        attention_mask=case["attention_mask"],
        output_hidden_states=True
    )

clean_states = clean_out.hidden_states

corrupt_emb = corrupted_embeddings(
    case["input_ids"],
    case["subject_positions"],
    12345,
    3.0
)

with torch.inference_mode():

    corrupt_out = model(
        inputs_embeds=corrupt_emb,
        attention_mask=case["attention_mask"]
    )

D0 = logit_margin(
    corrupt_out.logits,
    case["true_id"],
    case["new_id"]
)

Dpatch = patched_margin(
    case,
    clean_states,
    corrupt_emb,
    [
        (N_LAYERS - 1, "Q")
    ]
)

print("D clean:", case["D_clean"])
print("D corrupt:", D0)
print("D last-Q patch:", Dpatch)

D clean: 3.59515380859375
D corrupt: 1.0325927734375
D last-Q patch: 3.19195556640625


In [15]:
DISCOVERY_NOISE_SEED = 707
DISCOVERY_NOISE_SCALE = 3.0

N_SITES = len(SITES)
N_PAIRS = N_SITES * (N_SITES - 1) // 2

print("sites:", N_SITES)
print("pairs:", N_PAIRS)

sites: 24
pairs: 276


In [16]:
def runtime_case_discovery(case):

    with torch.inference_mode():

        clean_out = model(
            input_ids=case["input_ids"],
            attention_mask=case["attention_mask"],
            output_hidden_states=True
        )

    clean_states = clean_out.hidden_states

    D_clean = logit_margin(
        clean_out.logits,
        case["true_id"],
        case["new_id"]
    )

    actual_seed = (
        DISCOVERY_NOISE_SEED
        + int(case["case_id"])
    )

    corrupt_emb = corrupted_embeddings(
        case["input_ids"],
        case["subject_positions"],
        actual_seed,
        DISCOVERY_NOISE_SCALE
    )

    with torch.inference_mode():

        corrupt_out = model(
            inputs_embeds=corrupt_emb,
            attention_mask=case["attention_mask"]
        )

    D0 = logit_margin(
        corrupt_out.logits,
        case["true_id"],
        case["new_id"]
    )

    TE = D_clean - D0

    return (
        clean_states,
        corrupt_emb,
        D_clean,
        D0,
        TE
    )

In [17]:
SINGLE_PATH = (
    "phase7_single_discovery.csv"
)

PAIR_PATH = (
    "phase7_pair_discovery.csv"
)

In [18]:
if os.path.exists(SINGLE_PATH):

    single_df = pd.read_csv(
        SINGLE_PATH
    )

else:

    single_df = pd.DataFrame()


if os.path.exists(PAIR_PATH):

    pair_df = pd.read_csv(
        PAIR_PATH
    )

else:

    pair_df = pd.DataFrame()

In [19]:
complete_single = set()

if len(single_df):

    counts = (
        single_df
        .groupby("case_id")
        .size()
    )

    complete_single = set(
        counts[
            counts == N_SITES
        ].index
    )


complete_pair = set()

if len(pair_df):

    counts = (
        pair_df
        .groupby("case_id")
        .size()
    )

    complete_pair = set(
        counts[
            counts == N_PAIRS
        ].index
    )


complete_cases = (
    complete_single
    & complete_pair
)

print(
    "already complete:",
    len(complete_cases)
)

already complete: 9


In [20]:
import time

case = discovery_cases[0]

(
    clean_states,
    corrupt_emb,
    D_clean,
    D0,
    TE
) = runtime_case_discovery(case)

site = SITES[0]

t0 = time.perf_counter()

for _ in range(5):
    _ = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        [site]
    )

elapsed = time.perf_counter() - t0

sec_per_patch = elapsed / 5

print("seconds per patched forward:", sec_per_patch)
print(
    "estimated full scan hours:",
    sec_per_patch * 32 * 300 / 3600
)

seconds per patched forward: 0.20697280019521713
estimated full scan hours: 0.5519274671872457


In [21]:
single_rows = (
    single_df.to_dict("records")
    if len(single_df)
    else []
)

pair_rows = (
    pair_df.to_dict("records")
    if len(pair_df)
    else []
)


for ci, case in enumerate(
    discovery_cases
):

    case_id = case["case_id"]

    if case_id in complete_cases:

        print(
            f"{ci+1}/{len(discovery_cases)}",
            case["subject"],
            "| already complete"
        )

        continue

    (
        clean_states,
        corrupt_emb,
        D_clean,
        D0,
        TE
    ) = runtime_case_discovery(
        case
    )

    if TE <= 0:

        print(
            "WARNING: TE <= 0",
            case_id,
            TE
        )

        continue

    # --------------------------
    # Singles
    # --------------------------

    single_D = {}

    for site in SITES:

        Di = patched_margin(
            case,
            clean_states,
            corrupt_emb,
            [site]
        )

        single_D[site] = Di

        IE_norm = (
            Di - D0
        ) / TE

        single_rows.append({
            "case_id":
                case_id,

            "layer":
                site[0],

            "role":
                site[1],

            "D_clean":
                D_clean,

            "D0":
                D0,

            "TE":
                TE,

            "IE_norm":
                IE_norm
        })

    # --------------------------
    # Pairs
    # --------------------------

    for i in range(
        len(SITES)
    ):

        for j in range(
            i + 1,
            len(SITES)
        ):

            si = SITES[i]
            sj = SITES[j]

            Dij = patched_margin(
                case,
                clean_states,
                corrupt_emb,
                [si, sj]
            )

            Di = single_D[si]
            Dj = single_D[sj]

            J = (
                Dij
                - Di
                - Dj
                + D0
            ) / TE

            pair_recovery = (
                Dij - D0
            ) / TE

            pair_rows.append({
                "case_id":
                    case_id,

                "li":
                    si[0],

                "ri":
                    si[1],

                "lj":
                    sj[0],

                "rj":
                    sj[1],

                "J_norm":
                    J,

                "pair_recovery":
                    pair_recovery
            })

    # checkpoint dopo ogni fatto

    single_df = pd.DataFrame(
        single_rows
    )

    pair_df = pd.DataFrame(
        pair_rows
    )

    single_df.to_csv(
        SINGLE_PATH,
        index=False
    )

    pair_df.to_csv(
        PAIR_PATH,
        index=False
    )

    print(
        f"{ci+1}/{len(discovery_cases)}",
        case["subject"],
        "| TE =",
        round(TE, 3)
    )

1/32 Julien Courbet | TE = 2.905
2/32 Windows Mobile 6.5 | already complete
3/32 Brisbane Airport | TE = 5.423
4/32 Francis X. DiLorenzo | TE = 0.665
5/32 Nazi Party | already complete
6/32 Galina Starovoytova | TE = 1.861
7/32 Charles-Joseph Panckoucke | TE = 1.534
8/32 Saint-Jacques-de-la-Lande | TE = 4.283
9/32 The Kathmandu Post | TE = 1.633
11/32 Virginia | TE = 2.006
12/32 Broadway Open House | TE = 1.605
13/32 Microsoft Project | TE = 0.884
14/32 R. H. Bruce Lockhart | TE = 2.153
15/32 Honolulu International Airport | TE = 2.534
16/32 Hurrem sultan | TE = 1.541
17/32 New Zealand Labour Party | TE = 5.625
18/32 1977 Moscow bombings | TE = 3.124
19/32 Sommariva del Bosco | TE = 1.105
20/32 Google Person Finder | TE = 4.47
21/32 New Club, Edinburgh | TE = 5.37
22/32 Boeing Defense, Space & Security | TE = 2.355
23/32 Thomas Beaufort, Duke of Exeter | TE = 1.629
24/32 Michelangelo | TE = 1.824
25/32 Jerusalem syndrome | already complete
27/32 Bobby Taylor & the Vancouvers | already 

In [22]:
single_df = pd.read_csv(
    SINGLE_PATH
)

pair_df = pd.read_csv(
    PAIR_PATH
)

print(
    "single rows:",
    len(single_df)
)

print(
    "pair rows:",
    len(pair_df)
)

print(
    "single cases:",
    single_df["case_id"].nunique()
)

print(
    "pair cases:",
    pair_df["case_id"].nunique()
)

single rows: 2808
pair rows: 32292
single cases: 38
pair cases: 38


In [23]:
site_stats = (
    single_df
    .groupby(
        ["layer", "role"]
    )["IE_norm"]
    .agg(
        ["mean", "std", "count"]
    )
    .reset_index()
    .sort_values(
        "mean",
        ascending=False
    )
)

display(
    site_stats.head(15)
)

,layer,role,mean,std,count
20,10,Q,0.978765,0.087092,117
18,9,Q,0.937452,0.099573,117
16,8,Q,0.738802,0.230226,117
22,11,Q,0.592513,0.417057,117
5,2,S,0.418584,0.448530,117
9,4,S,0.411854,0.412893,117
14,7,Q,0.405963,0.349343,117
3,1,S,0.345287,0.344428,117
13,6,S,0.344982,0.427573,117
7,3,S,0.343567,0.381324,117


In [24]:
AIE_PAIR_PYTHIA = [
    (
        int(row.layer),
        row.role
    )
    for row in (
        site_stats
        .head(2)
        .itertuples()
    )
]

print(
    "AIE pair:",
    AIE_PAIR_PYTHIA
)

AIE pair: [(10, 'Q'), (9, 'Q')]


In [25]:
pair_stats = (
    pair_df
    .groupby(
        [
            "li",
            "ri",
            "lj",
            "rj"
        ]
    )
    .agg(
        mean_J=(
            "J_norm",
            "mean"
        ),

        std_J=(
            "J_norm",
            "std"
        ),

        mean_pair_recovery=(
            "pair_recovery",
            "mean"
        ),

        count=(
            "pair_recovery",
            "count"
        )
    )
    .reset_index()
)

In [26]:
display(
    pair_stats
    .sort_values(
        "mean_J",
        ascending=False
    )
    .head(15)
)

,li,ri,lj,rj,mean_J,std_J,mean_pair_recovery,count
188,5,Q,7,S,0.076846,0.367129,0.620820,117
197,5,S,5,Q,0.059999,0.286010,0.621055,117
139,3,S,3,Q,0.059546,0.349406,0.499563,117
186,5,Q,6,S,0.048092,0.279580,0.613037,117
130,3,Q,7,S,0.022533,0.218954,0.442995,117
239,7,S,7,Q,0.019258,0.319831,0.749232,117
87,2,Q,3,S,0.016592,0.234546,0.435540,117
17,0,Q,9,S,0.014110,0.132381,0.072255,117
126,3,Q,5,S,0.012712,0.264306,0.450255,117
190,5,Q,8,S,0.012556,0.358561,0.441518,117


In [27]:
best_pairs = (
    pair_stats
    .sort_values(
        "mean_pair_recovery",
        ascending=False
    )
)

display(
    best_pairs.head(15)
)

,li,ri,lj,rj,mean_J,std_J,mean_pair_recovery,count
181,4,S,10,Q,-0.402544,0.408738,0.988075,117
206,5,S,10,Q,-0.334092,0.437980,0.985766,117
152,3,S,10,Q,-0.337406,0.381047,0.984926,117
227,6,S,10,Q,-0.339094,0.421358,0.984653,117
244,7,S,10,Q,-0.321221,0.395049,0.981555,117
266,9,S,10,Q,-0.009629,0.062494,0.980678,117
272,10,S,10,Q,-0.000548,0.012907,0.980510,117
271,10,Q,11,S,0.000000,0.000000,0.978765,117
100,2,Q,10,Q,-0.075381,0.233083,0.978765,117
61,1,Q,10,Q,-0.077419,0.220502,0.978765,117


In [28]:
best = best_pairs.iloc[0]

INT_PAIR_PYTHIA = [
    (
        int(best["li"]),
        best["ri"]
    ),

    (
        int(best["lj"]),
        best["rj"]
    )
]

print(
    "AIE pair:",
    AIE_PAIR_PYTHIA
)

print(
    "Interaction-aware pair:",
    INT_PAIR_PYTHIA
)

print(
    "Discovery mean recovery:",
    best["mean_pair_recovery"]
)

print(
    "Mean J:",
    best["mean_J"]
)

AIE pair: [(10, 'Q'), (9, 'Q')]
Interaction-aware pair: [(4, 'S'), (10, 'Q')]
Discovery mean recovery: 0.9880747795099856
Mean J: -0.4025440893727636


In [29]:
def canonical_pair(a, b):

    return tuple(
        sorted(
            [a, b]
        )
    )

In [30]:
def get_pair_rows(
    pair_df,
    pair
):

    a, b = sorted(pair)

    mask = (
        (pair_df["li"] == a[0])
        &
        (pair_df["ri"] == a[1])
        &
        (pair_df["lj"] == b[0])
        &
        (pair_df["rj"] == b[1])
    )

    return pair_df[
        mask
    ].copy()

In [31]:
aie_disc = get_pair_rows(
    pair_df,
    AIE_PAIR_PYTHIA
)

int_disc = get_pair_rows(
    pair_df,
    INT_PAIR_PYTHIA
)

In [32]:
discovery_compare = (
    aie_disc[
        [
            "case_id",
            "pair_recovery"
        ]
    ]
    .rename(
        columns={
            "pair_recovery":
                "recovery_AIE"
        }
    )
    .merge(
        int_disc[
            [
                "case_id",
                "pair_recovery"
            ]
        ].rename(
            columns={
                "pair_recovery":
                    "recovery_INT"
            }
        ),
        on="case_id"
    )
)

discovery_compare[
    "delta"
] = (
    discovery_compare[
        "recovery_INT"
    ]
    -
    discovery_compare[
        "recovery_AIE"
    ]
)

print(
    discovery_compare[
        [
            "recovery_AIE",
            "recovery_INT",
            "delta"
        ]
    ].mean()
)

recovery_AIE    0.980480
recovery_INT    0.992615
delta           0.012135
dtype: float64


In [33]:
single_key = [
    "case_id", "layer", "role"
]

pair_key = [
    "case_id",
    "li", "ri",
    "lj", "rj"
]

print(
    "single duplicate rows:",
    single_df.duplicated(
        single_key,
        keep=False
    ).sum()
)

print(
    "pair duplicate rows:",
    pair_df.duplicated(
        pair_key,
        keep=False
    ).sum()
)

single duplicate rows: 2592
pair duplicate rows: 29808


In [34]:
single_consistency = (
    single_df
    .groupby(single_key)["IE_norm"]
    .agg(lambda x: x.max() - x.min())
)

pair_J_consistency = (
    pair_df
    .groupby(pair_key)["J_norm"]
    .agg(lambda x: x.max() - x.min())
)

pair_R_consistency = (
    pair_df
    .groupby(pair_key)["pair_recovery"]
    .agg(lambda x: x.max() - x.min())
)

print(
    "max single discrepancy:",
    single_consistency.max()
)

print(
    "max pair J discrepancy:",
    pair_J_consistency.max()
)

print(
    "max pair recovery discrepancy:",
    pair_R_consistency.max()
)

max single discrepancy: 1.133839779005525
max pair J discrepancy: 2.8005524861878452
max pair recovery discrepancy: 1.364364640883978


In [35]:
single_clean = (
    single_df
    .drop_duplicates(
        subset=single_key,
        keep="first"
    )
    .copy()
)

pair_clean = (
    pair_df
    .drop_duplicates(
        subset=pair_key,
        keep="first"
    )
    .copy()
)

print(
    "single clean:",
    len(single_clean)
)

print(
    "pair clean:",
    len(pair_clean)
)

print(
    "cases:",
    single_clean["case_id"].nunique(),
    pair_clean["case_id"].nunique()
)

single clean: 912
pair clean: 10488
cases: 38 38


In [36]:
single_clean.to_csv(
    "phase7_single_discovery_clean.csv",
    index=False
)

pair_clean.to_csv(
    "phase7_pair_discovery_clean.csv",
    index=False
)

In [37]:
expected_ids = {
    c["case_id"]
    for c in discovery_cases
}

completed_ids = set(
    single_clean["case_id"]
)

missing_ids = (
    expected_ids
    - completed_ids
)

print("missing:", len(missing_ids))

for c in discovery_cases:
    if c["case_id"] in missing_ids:
        print(
            c["case_id"],
            c["subject"]
        )

missing: 2
16862 Economic Community of West African States
14502 Montserrat Lombard


In [38]:
single_clean.to_csv(
    "phase7_single_discovery_clean.csv",
    index=False
)

pair_clean.to_csv(
    "phase7_pair_discovery_clean.csv",
    index=False
)

In [39]:
excluded_discovery_ids = {
    9810,
    10921,
    4770,
}

excluded_rows = []

for c in discovery_cases:

    if c["case_id"] in excluded_discovery_ids:

        (
            _,
            _,
            D_clean,
            D0,
            TE
        ) = runtime_case_discovery(c)

        excluded_rows.append({
            "case_id": c["case_id"],
            "subject": c["subject"],
            "D_clean": D_clean,
            "D0": D0,
            "TE": TE,
            "reason": "discovery TE <= 0"
        })

pd.DataFrame(
    excluded_rows
).to_csv(
    "phase7_discovery_excluded.csv",
    index=False
)

pd.DataFrame(excluded_rows)

,case_id,subject,D_clean,D0,TE,reason
0,9810,Windows Mobile 6.5,1.853027,0.543274,1.309753,discovery TE <= 0


In [40]:
rng = np.random.default_rng(
    DISCOVERY_SEED
)

indices = rng.permutation(
    len(counterfact)
)

In [41]:
original_discovery_subjects = {
    norm_subject(c["subject"])
    for c in discovery_cases
}

used_subjects = (
    PREVIOUS_SUBJECTS
    | original_discovery_subjects
)

In [42]:
replacement_cases = []

for idx in indices:

    record = counterfact[int(idx)]

    subject_key = norm_subject(
        record[
            "requested_rewrite"
        ]["subject"]
    )

    # niente soggetti già visti
    if subject_key in used_subjects:
        continue

    case = prepare_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    (
        _,
        _,
        D_clean,
        D0,
        TE
    ) = runtime_case_discovery(
        case
    )

    # stesso identico criterio del loop originale
    if TE <= 0:
        continue

    replacement_cases.append(
        case
    )

    used_subjects.add(
        subject_key
    )

    print(
        len(replacement_cases),
        case["case_id"],
        case["subject"],
        "| discovery TE =",
        round(TE, 3)
    )

    if len(replacement_cases) == 3:
        break

1 13631 Toke Makinwa | discovery TE = 0.832
2 17582 Wagram Music | discovery TE = 1.669
3 19349 Lewin Brzeski | discovery TE = 1.975


In [43]:
assert len(replacement_cases) == 3

assert len({
    norm_subject(c["subject"])
    for c in replacement_cases
}) == 3

In [44]:
single_final = (
    single_clean.copy()
)

pair_final = (
    pair_clean.copy()
)

In [45]:
for ci, case in enumerate(
    replacement_cases,
    start=1
):

    case_id = case["case_id"]

    (
        clean_states,
        corrupt_emb,
        D_clean,
        D0,
        TE
    ) = runtime_case_discovery(
        case
    )

    assert TE > 0

    # -----------------
    # Singles
    # -----------------

    single_D = {}

    new_single_rows = []

    for site in SITES:

        Di = patched_margin(
            case,
            clean_states,
            corrupt_emb,
            [site]
        )

        single_D[site] = Di

        new_single_rows.append({
            "case_id": case_id,
            "layer": site[0],
            "role": site[1],
            "D_clean": D_clean,
            "D0": D0,
            "TE": TE,
            "IE_norm": (
                Di - D0
            ) / TE
        })

    # -----------------
    # Pairs
    # -----------------

    new_pair_rows = []

    for i in range(len(SITES)):

        for j in range(
            i + 1,
            len(SITES)
        ):

            si = SITES[i]
            sj = SITES[j]

            Dij = patched_margin(
                case,
                clean_states,
                corrupt_emb,
                [si, sj]
            )

            Di = single_D[si]
            Dj = single_D[sj]

            J = (
                Dij
                - Di
                - Dj
                + D0
            ) / TE

            recovery = (
                Dij - D0
            ) / TE

            new_pair_rows.append({
                "case_id": case_id,
                "li": si[0],
                "ri": si[1],
                "lj": sj[0],
                "rj": sj[1],
                "J_norm": J,
                "pair_recovery": recovery
            })

    single_final = pd.concat(
        [
            single_final,
            pd.DataFrame(
                new_single_rows
            )
        ],
        ignore_index=True
    )

    pair_final = pd.concat(
        [
            pair_final,
            pd.DataFrame(
                new_pair_rows
            )
        ],
        ignore_index=True
    )

    # checkpoint dopo ciascun sostituto
    single_final.to_csv(
        "phase7_single_discovery_final.csv",
        index=False
    )

    pair_final.to_csv(
        "phase7_pair_discovery_final.csv",
        index=False
    )

    print(
        f"replacement {ci}/3 complete:",
        case["subject"]
    )

replacement 1/3 complete: Toke Makinwa
replacement 2/3 complete: Wagram Music
replacement 3/3 complete: Lewin Brzeski


In [46]:
print(
    "single rows:",
    len(single_final)
)

print(
    "pair rows:",
    len(pair_final)
)

print(
    "single cases:",
    single_final[
        "case_id"
    ].nunique()
)

print(
    "pair cases:",
    pair_final[
        "case_id"
    ].nunique()
)

print(
    "single duplicates:",
    single_final.duplicated(
        single_key
    ).sum()
)

print(
    "pair duplicates:",
    pair_final.duplicated(
        pair_key
    ).sum()
)

single rows: 984
pair rows: 11316
single cases: 41
pair cases: 41
single duplicates: 0
pair duplicates: 0


In [47]:
completed_ids = set(
    single_clean["case_id"]
)

retained_cases = [
    c
    for c in discovery_cases
    if c["case_id"] in completed_ids
]

In [48]:
discovery_cases_final = (
    retained_cases
    + replacement_cases
)

assert len(
    discovery_cases_final
) == 32

assert len({
    norm_subject(c["subject"])
    for c in discovery_cases_final
}) == 32

AssertionError: 

In [ ]:
discovery_manifest_final = pd.DataFrame([
    {
        "case_id": c["case_id"],
        "dataset_index": c["dataset_index"],
        "subject": c["subject"],
        "prompt": c["prompt"],
        "true_text": c["true_text"],
        "new_text": c["new_text"],
        "D_clean": c["D_clean"],
        "TE_reference": c["TE_reference"]
    }
    for c in discovery_cases_final
])

discovery_manifest_final.to_csv(
    "phase7_discovery_manifest_final.csv",
    index=False
)

In [ ]:
site_stats = (
    single_final
    .groupby(
        ["layer", "role"]
    )["IE_norm"]
    .agg(
        ["mean", "std", "count"]
    )
    .reset_index()
    .sort_values(
        "mean",
        ascending=False
    )
)

display(
    site_stats.head(15)
)

In [ ]:
pair_stats = (
    pair_final
    .groupby(
        [
            "li", "ri",
            "lj", "rj"
        ]
    )
    .agg(
        mean_J=(
            "J_norm",
            "mean"
        ),
        std_J=(
            "J_norm",
            "std"
        ),
        mean_pair_recovery=(
            "pair_recovery",
            "mean"
        ),
        count=(
            "pair_recovery",
            "count"
        )
    )
    .reset_index()
)

In [ ]:
display(
    pair_stats
    .sort_values(
        "mean_J",
        ascending=False
    )
    .head(15)
)

In [ ]:
best_pairs = (
    pair_stats
    .sort_values(
        "mean_pair_recovery",
        ascending=False
    )
)

display(
    best_pairs.head(15)
)

In [ ]:
AIE_PAIR_PYTHIA = [
    (
        int(row.layer),
        row.role
    )
    for row in (
        site_stats
        .head(2)
        .itertuples()
    )
]

best = best_pairs.iloc[0]

INT_PAIR_PYTHIA = [
    (
        int(best["li"]),
        best["ri"]
    ),
    (
        int(best["lj"]),
        best["rj"]
    )
]

print(
    "AIE pair:",
    AIE_PAIR_PYTHIA
)

print(
    "INT pair:",
    INT_PAIR_PYTHIA
)

print(
    "INT mean recovery:",
    best[
        "mean_pair_recovery"
    ]
)

print(
    "INT mean J:",
    best[
        "mean_J"
    ]
)

In [ ]:
AIE_PAIR_PYTHIA = [
    (10, "Q"),
    (9, "Q"),
]

INT_PAIR_PYTHIA = [
    (4, "S"),
    (10, "Q"),
]

In [ ]:
def get_pair_rows(pair_df, pair):

    a, b = sorted(pair)

    mask = (
        (pair_df["li"] == a[0])
        &
        (pair_df["ri"] == a[1])
        &
        (pair_df["lj"] == b[0])
        &
        (pair_df["rj"] == b[1])
    )

    return pair_df[
        mask
    ].copy()

In [ ]:
aie_disc = get_pair_rows(
    pair_final,
    AIE_PAIR_PYTHIA
)

int_disc = get_pair_rows(
    pair_final,
    INT_PAIR_PYTHIA
)

compare = (
    aie_disc[
        ["case_id", "pair_recovery"]
    ]
    .rename(
        columns={
            "pair_recovery":
            "recovery_AIE"
        }
    )
    .merge(
        int_disc[
            ["case_id", "pair_recovery"]
        ]
        .rename(
            columns={
                "pair_recovery":
                "recovery_INT"
            }
        ),
        on="case_id"
    )
)

compare["delta"] = (
    compare["recovery_INT"]
    -
    compare["recovery_AIE"]
)

print(compare.describe())

print(
    "\nMean AIE:",
    compare["recovery_AIE"].mean()
)

print(
    "Mean INT:",
    compare["recovery_INT"].mean()
)

print(
    "Mean delta:",
    compare["delta"].mean()
)

print(
    "INT win rate:",
    (compare["delta"] > 0).mean()
)

In [ ]:
import hashlib
import json

with open(
    "phase7_discovery_manifest_final.csv",
    "rb"
) as f:

    discovery_hash = (
        hashlib.sha256(
            f.read()
        ).hexdigest()
    )

In [ ]:
phase7_selection = {
    "model":
        MODEL_ID,

    "discovery_n":
        32,

    "discovery_manifest_sha256":
        discovery_hash,

    "AIE_pair":
        AIE_PAIR_PYTHIA,

    "interaction_pair":
        INT_PAIR_PYTHIA,

    "selection_rule_AIE":
        "top two sites by mean normalized single-site IE",

    "selection_rule_interaction":
        "pair with maximum mean measured pair recovery",

    "primary_holdout_metric":
        "normalized causal recovery",

    "primary_success":
        "paired bootstrap 95% CI lower bound > 0",

    "practical_threshold":
        0.02,

    "noise_scales":
        [2.5, 3.0, 3.5],

    "noise_seeds":
        [101, 202, 303],

    "primary_scale":
        3.0
}

In [ ]:
with open(
    "phase7_frozen_selection.json",
    "w"
) as f:

    json.dump(
        phase7_selection,
        f,
        indent=2
    )

In [ ]:
DISCOVERY_SUBJECTS = {
    norm_subject(c["subject"])
    for c in discovery_cases_final
}

ALL_SEEN_SUBJECTS = (
    PREVIOUS_SUBJECTS
    | DISCOVERY_SUBJECTS
)

print(
    "subjects excluded from holdout:",
    len(ALL_SEEN_SUBJECTS)
)

In [ ]:
rng_holdout = np.random.default_rng(
    HOLDOUT_SEED
)

indices_holdout = (
    rng_holdout.permutation(
        len(counterfact)
    )
)

holdout_cases = []
holdout_subjects = set()

for idx in indices_holdout:

    record = counterfact[
        int(idx)
    ]

    subject_key = norm_subject(
        record[
            "requested_rewrite"
        ]["subject"]
    )

    if subject_key in ALL_SEEN_SUBJECTS:
        continue

    if subject_key in holdout_subjects:
        continue

    case = prepare_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    holdout_cases.append(
        case
    )

    holdout_subjects.add(
        subject_key
    )

    print(
        len(holdout_cases),
        case["subject"],
        "| reference TE =",
        round(
            case["TE_reference"],
            3
        )
    )

    if len(holdout_cases) == 128:
        break

In [ ]:
assert len(
    holdout_cases
) == 128

assert len(
    holdout_subjects
) == 128

assert (
    holdout_subjects
    & ALL_SEEN_SUBJECTS
) == set()

print(
    "holdout:",
    len(holdout_cases)
)

print(
    "unique subjects:",
    len(holdout_subjects)
)

print(
    "previous-subject overlap:",
    len(
        holdout_subjects
        & ALL_SEEN_SUBJECTS
    )
)

In [ ]:
holdout_manifest = pd.DataFrame([
    {
        "case_id": c["case_id"],
        "dataset_index": c["dataset_index"],
        "subject": c["subject"],
        "prompt": c["prompt"],
        "true_text": c["true_text"],
        "new_text": c["new_text"],
        "D_clean": c["D_clean"],
        "TE_reference": c["TE_reference"],
    }
    for c in holdout_cases
])

holdout_manifest.to_csv(
    "phase7_holdout_manifest.csv",
    index=False
)

In [ ]:
import hashlib

with open(
    "phase7_holdout_manifest.csv",
    "rb"
) as f:

    HOLDOUT_HASH = hashlib.sha256(
        f.read()
    ).hexdigest()

print(HOLDOUT_HASH)

In [ ]:
phase7_protocol = {
    "experiment":
        "Phase 7 Pythia cross-model replication",

    "model":
        MODEL_ID,

    "discovery_n":
        32,

    "holdout_n":
        128,

    "AIE_pair":
        AIE_PAIR_PYTHIA,

    "interaction_pair":
        INT_PAIR_PYTHIA,

    "holdout_manifest_sha256":
        HOLDOUT_HASH,

    "primary_scale":
        3.0,

    "noise_scales":
        [2.5, 3.0, 3.5],

    "noise_seeds":
        [101, 202, 303],

    "min_eval_TE":
        0.5,

    "primary_metric":
        "normalized causal recovery",

    "primary_success":
        "paired bootstrap 95% CI lower bound > 0",

    "practical_threshold":
        0.02,

    "note":
        (
            "Discovery mean advantage was 0.0078125; "
            "interaction pair won 1/32 discovery cases."
        )
}

with open(
    "phase7_frozen_protocol.json",
    "w"
) as f:

    json.dump(
        phase7_protocol,
        f,
        indent=2
    )

In [ ]:
def evaluate_holdout_condition(
    case,
    noise_multiplier,
    noise_seed
):

    with torch.inference_mode():

        clean_out = model(
            input_ids=
                case["input_ids"],

            attention_mask=
                case["attention_mask"],

            output_hidden_states=True
        )

    clean_states = (
        clean_out.hidden_states
    )

    D_clean = logit_margin(
        clean_out.logits,
        case["true_id"],
        case["new_id"]
    )

    actual_seed = (
        int(noise_seed)
        +
        int(case["case_id"])
    )

    corrupt_emb = (
        corrupted_embeddings(
            case["input_ids"],
            case["subject_positions"],
            actual_seed,
            noise_multiplier
        )
    )

    with torch.inference_mode():

        corrupt_out = model(
            inputs_embeds=
                corrupt_emb,

            attention_mask=
                case["attention_mask"]
        )

    D_corr = logit_margin(
        corrupt_out.logits,
        case["true_id"],
        case["new_id"]
    )

    TE = (
        D_clean - D_corr
    )

    D_AIE = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        AIE_PAIR_PYTHIA
    )

    D_INT = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        INT_PAIR_PYTHIA
    )

    valid = (
        TE >= MIN_EVAL_TE
    )

    if valid:

        recovery_AIE = (
            D_AIE - D_corr
        ) / TE

        recovery_INT = (
            D_INT - D_corr
        ) / TE

        delta = (
            recovery_INT
            - recovery_AIE
        )

    else:

        recovery_AIE = np.nan
        recovery_INT = np.nan
        delta = np.nan

    return {
        "case_id":
            case["case_id"],

        "noise_scale":
            noise_multiplier,

        "noise_seed":
            noise_seed,

        "D_clean":
            D_clean,

        "D_corr":
            D_corr,

        "TE":
            TE,

        "valid":
            valid,

        "D_AIE":
            D_AIE,

        "D_INT":
            D_INT,

        "recovery_AIE":
            recovery_AIE,

        "recovery_INT":
            recovery_INT,

        "delta_recovery":
            delta,

        "margin_advantage_INT":
            D_INT - D_AIE,

        "correct_AIE":
            float(D_AIE > 0),

        "correct_INT":
            float(D_INT > 0),

        "manifest_sha256":
            HOLDOUT_HASH
    }

In [ ]:
RESULT_PATH = (
    "phase7_holdout_results.csv"
)

if os.path.exists(
    RESULT_PATH
):

    phase7_results = pd.read_csv(
        RESULT_PATH
    )

    hashes = set(
        phase7_results[
            "manifest_sha256"
        ]
        .dropna()
        .astype(str)
    )

    if hashes != {
        HOLDOUT_HASH
    }:

        raise RuntimeError(
            "Result file belongs "
            "to another manifest."
        )

else:

    phase7_results = (
        pd.DataFrame()
    )

In [ ]:
done = set()

if len(
    phase7_results
):

    done = set(
        zip(
            phase7_results[
                "case_id"
            ],

            phase7_results[
                "noise_scale"
            ],

            phase7_results[
                "noise_seed"
            ]
        )
    )

print(
    "already complete:",
    len(done)
)

In [ ]:
rows = (
    phase7_results
    .to_dict("records")
    if len(phase7_results)
    else []
)

for ci, case in enumerate(
    holdout_cases
):

    for scale in NOISE_SCALES:

        for seed in NOISE_SEEDS:

            key = (
                case["case_id"],
                scale,
                seed
            )

            if key in done:
                continue

            row = (
                evaluate_holdout_condition(
                    case,
                    scale,
                    seed
                )
            )

            rows.append(row)
            done.add(key)

            pd.DataFrame(
                rows
            ).to_csv(
                RESULT_PATH,
                index=False
            )

    print(
        f"{ci+1}/128",
        case["subject"]
    )

phase7_results = (
    pd.DataFrame(rows)
)

In [ ]:
primary = phase7_results[
    (
        phase7_results[
            "noise_scale"
        ] == 3.0
    )
    &
    phase7_results[
        "valid"
    ]
]

In [ ]:
primary_case = (
    primary
    .groupby("case_id")
    .agg(
        recovery_AIE=(
            "recovery_AIE",
            "mean"
        ),

        recovery_INT=(
            "recovery_INT",
            "mean"
        ),

        delta=(
            "delta_recovery",
            "mean"
        ),

        margin_advantage=(
            "margin_advantage_INT",
            "mean"
        ),

        n_valid_seeds=(
            "noise_seed",
            "nunique"
        )
    )
)

In [ ]:
def bootstrap_mean_ci(
    values,
    n_boot=10000,
    seed=12345
):

    x = np.asarray(
        values,
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    rng = (
        np.random.default_rng(
            seed
        )
    )

    n = len(x)

    boot = np.empty(
        n_boot
    )

    for b in range(
        n_boot
    ):

        sample = rng.choice(
            x,
            size=n,
            replace=True
        )

        boot[b] = (
            sample.mean()
        )

    return (
        x.mean(),
        np.quantile(
            boot,
            0.025
        ),
        np.quantile(
            boot,
            0.975
        )
    )

In [ ]:
delta_mean, lo, hi = (
    bootstrap_mean_ci(
        primary_case[
            "delta"
        ]
    )
)

print(
    "n primary:",
    len(primary_case)
)

print(
    "AIE:",
    primary_case[
        "recovery_AIE"
    ].mean()
)

print(
    "INT:",
    primary_case[
        "recovery_INT"
    ].mean()
)

print(
    "delta:",
    delta_mean
)

print(
    "95% CI:",
    lo,
    hi
)

print(
    "statistical success:",
    lo > 0
)

print(
    "practical success:",
    delta_mean >= 0.02
)

print(
    "win rate:",
    (
        primary_case["delta"]
        > 0
    ).mean()
)

print(
    "tie rate:",
    np.isclose(
        primary_case["delta"],
        0
    ).mean()
)

In [ ]:
# Ultimi controlli
complete_ids = primary_case[
    primary_case["n_valid_seeds"] == 3
].index

complete = primary_case.loc[complete_ids]

mean3, lo3, hi3 = bootstrap_mean_ci(
    complete["delta"]
)

print("Complete 3-seed cases:", len(complete))
print("delta:", mean3)
print("95% CI:", lo3, hi3)

In [ ]:
raw_primary = phase7_results[
    phase7_results["noise_scale"] == PRIMARY_SCALE
]

raw_by_case = (
    raw_primary
    .groupby("case_id")["margin_advantage_INT"]
    .mean()
)

raw_mean, raw_lo, raw_hi = bootstrap_mean_ci(
    raw_by_case
)

print("Raw-margin cases:", len(raw_by_case))
print("mean margin advantage:", raw_mean)
print("95% CI:", raw_lo, raw_hi)